<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/Computer-Science/Operating-Systems/05-synchronization-correctness-and-liveness.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Operating Systems guideline](Operating-Systems.html)


## **Synchronization, Concurrent Correctness, and Liveness**

The scheduler from the previous chapter may stop a thread at almost any legal preemption point, and several processors may execute shared-memory instructions at the same time. **Synchronization** is the set of protocols and mechanisms that makes selected relationships among those executions explicit. It protects state invariants, communicates when a condition may have changed, and constrains memory visibility and ordering.

Synchronization is not synonymous with adding a mutex around every access. A correct design first identifies the shared object, the invariant that makes its states meaningful, the operations that must appear indivisible, and the progress expected from competing threads. It then chooses an appropriate mechanism: exclusive ownership, atomic operations, condition synchronization, phase coordination, versioned reads, or isolation that avoids shared memory entirely.

The recurring shell pipeline already contains synchronization:

```bash
cat input.txt | grep kernel > result.txt
```

`cat` and `grep` normally live in separate address spaces, so they cannot accidentally race on ordinary heap variables. The kernel pipe is their shared bounded channel. A write may block while the pipe is full; a read may block while it is empty; closing the final write end makes end-of-file observable. If the same pipeline were implemented as threads around an in-process ring buffer, the application would need to preserve those state and wakeup rules itself.

### **Why Concurrent Programs Are Difficult**

A sequential function can often be understood as a path from input state to output state. A concurrent component must be correct for every execution permitted by the language, synchronization protocol, operating system, compiler, and hardware memory model. Bugs hide in rare orderings, disappear when logging changes timing, and may occur only on a different processor architecture or CPU count.

The source code is not the execution schedule. A single expression may compile into several loads, arithmetic instructions, and stores. The compiler may reorder operations when the language allows it. Hardware may execute and propagate independent memory operations out of source order. Cache coherence keeps processors from permanently disagreeing about one cache line, but it does not make a multi-step protocol atomic or impose every cross-location ordering the programmer imagined.

#### **Interleavings and Nondeterminism**

An **interleaving** is one ordering of operations from several threads that preserves each thread's own required order. If thread A has $m$ indivisible steps and thread B has $n$, even without loops there can be

$$
\binom{m+n}{m}=\frac{(m+n)!}{m!n!}
$$

relative interleavings. Two three-step operations already permit 20 schedules. More threads, branches, interrupts, weak ordering, and retries make exhaustive testing rapidly infeasible.

Consider two threads executing `counter++` from `counter = 0`. At the source level the operation looks singular, but a simplified implementation is:

```text
register = load(counter)
register = register + 1
store(counter, register)
```

![Two threads load the same counter value and both store one, losing one update.](assets/lost-update-interleaving-animated.svg){fig-alt="Animated instruction timeline showing two counter increments interleaving and producing one instead of two." width="98%"}

*Figure: original explanatory diagram created for this chapter, following the uncontrolled-scheduling example in [Operating Systems: Three Easy Pieces, Concurrency](https://pages.cs.wisc.edu/~remzi/OSTEP/threads-intro.pdf).*

Every instruction in the diagram succeeds. The failure is in the composition: both threads make a decision from the same stale snapshot. Other schedules happen to produce `2`, which is why repeated success does not establish correctness.

Concurrency is **nondeterministic** because the program admits more than one legal execution; it need not involve a random-number generator. Timing depends on preemption, interrupts, cache misses, page faults, I/O completion, and other load. Adding `printf`, a debugger, or a sleep changes those events and can hide the failing schedule. Correctness must come from an ordering protocol, not a timing expectation such as "thread A will probably finish first."

#### **Safety and Liveness Properties**

Concurrent correctness has two complementary dimensions:

- A **safety property** says that something bad never happens. Examples include "at most one thread owns this mutex," "the queue count remains between zero and capacity," and "a reader never dereferences freed storage."
- A **liveness property** says that something good eventually happens under stated assumptions. Examples include "a released mutex is eventually acquired by a waiter" and "every enqueued request eventually completes."

Safety is usually expressed through an **invariant**, a predicate that holds initially and remains true across every atomic state transition. For a bounded queue, a representation invariant might be:

$$
0 \le \text{count} \le \text{capacity},
$$

with `head`, `tail`, and occupied slots agreeing with `count`. A lock is useful only because all operations that can temporarily violate this invariant follow the same locking rule.

Liveness needs environmental assumptions. A thread cannot guarantee progress if the scheduler never runs it, hardware fails, or a lock owner never returns. Common strengths include:

| Property | Informal guarantee |
|---|---|
| deadlock freedom | the system as a whole does not stop because of a wait cycle |
| starvation freedom | every continuously eligible participant eventually succeeds |
| bounded waiting | a participant is overtaken at most a bounded number of times |
| lock freedom | some operation completes despite stalled participants |
| wait freedom | every operation completes within a bounded number of its own steps |

These are not interchangeable. A spinlock can preserve mutual exclusion while starving one waiter. A system can be deadlock-free while one unlucky request never completes. A timeout can return control without restoring the higher-level operation's liveness.

For concurrent objects, **linearizability** is a useful safety criterion. Each completed operation should appear to take effect at one instant between invocation and response, while respecting real-time order between non-overlapping calls. That conceptual instant is the **linearization point**. For an atomic increment it may be the hardware read-modify-write; for a mutex-protected queue insertion it is a state update inside the critical section. Linearizability does not by itself guarantee fairness or termination.

### **Race Conditions and Critical Sections**

A **race condition** exists when program correctness depends on the relative timing of concurrent events. A **data race** has a narrower language-level meaning: conflicting accesses to the same memory location occur concurrently, at least one is a write, and the accesses lack the required synchronization. In C, a data race on a non-atomic object produces undefined behavior. A high-level race condition can still exist without a data race, for example when two atomic "check then act" steps are individually safe but not combined into one transaction.

A **critical section** is code that accesses shared state under a protocol that prevents forbidden interleavings. Mutual exclusion typically aims for:

1. at most one owner is in the critical section;
2. a thread outside the critical section cannot prevent all others forever;
3. waiting behavior satisfies the intended fairness or bound;
4. entry and exit also provide the required memory visibility.

The following program is intentionally incorrect:

<details>
<summary><strong>C: an intentional pthread data race</strong></summary>

```c
#include <pthread.h>
#include <stdio.h>

enum { ITERATIONS = 1000000 };
static long counter = 0;

static void *increment(void *unused) {
    (void)unused;
    for (long i = 0; i < ITERATIONS; ++i) {
        counter++;  // Data race: read-modify-write is not synchronized.
    }
    return NULL;
}

int main(void) {
    pthread_t first;
    pthread_t second;

    if (pthread_create(&first, NULL, increment, NULL) != 0) {
        fputs("first pthread_create failed\n", stderr);
        return 1;
    }

    if (pthread_create(&second, NULL, increment, NULL) != 0) {
        fputs("second pthread_create failed\n", stderr);
        pthread_join(first, NULL);
        return 1;
    }

    int first_join = pthread_join(first, NULL);
    int second_join = pthread_join(second, NULL);
    if (first_join != 0 || second_join != 0) {
        fputs("pthread_join failed\n", stderr);
        return 1;
    }
    printf("counter = %ld; expected = %d\n", counter, 2 * ITERATIONS);
    return 0;
}
```

Because the C program has undefined behavior, no particular wrong result is promised. Observing `2000000` does not make it correct, and observing a smaller number is only one possible symptom.

</details>

The first repair question is not "which line needs a lock?" but "what operation and invariant must be atomic?" If the counter represents an independent statistic, an atomic fetch-add may be sufficient. If changing it must agree with a queue, balance, or ownership table, one mutex may need to protect the entire multi-field transition.

#### **Atomicity, Visibility, and Ordering**

Three requirements are often confused:

| Requirement | Question | Typical mechanism |
|---|---|---|
| atomicity | can another observer see an intermediate update? | mutex, atomic read-modify-write, transaction |
| visibility | when does another thread observe the update? | lock handoff, release/acquire, condition protocol |
| ordering | which operations must be observed before which others? | language memory order, barriers, synchronization operations |

An aligned machine-word store may be indivisible yet still be insufficient. Publishing a pointer atomically before the pointed-to object is initialized creates an ordering bug. Protecting only the pointer with a mutex while readers access object fields without the same protocol creates a visibility and lifetime bug. Correctness is a property of the entire communication path.

A mutex-based counter makes the compound operation atomic and creates a synchronization edge between unlock and a later successful lock:

```c
static pthread_mutex_t counter_lock = PTHREAD_MUTEX_INITIALIZER;
static long counter;

void increment_counter(void) {
    pthread_mutex_lock(&counter_lock);
    counter++;
    pthread_mutex_unlock(&counter_lock);
}
```

The lock should be documented as protecting `counter` and every invariant coupled to it. Accessing the same state elsewhere without the lock invalidates the proof. A naming convention such as `counter_lock` helps, but executable assertions and encapsulation are stronger than comments alone.

`volatile` is not a replacement. In C it constrains certain compiler treatment of individual accesses, especially for memory-mapped I/O and signal-related cases. It does not make `counter++` atomic, establish inter-thread happens-before, or provide mutual exclusion.

#### **The Memory Model**

A programming-language **memory model** defines which values concurrent reads may observe and which reorderings an implementation may expose. Portable reasoning must use that model rather than assumptions learned from one processor. The C model provides atomic types, memory orders, and a **happens-before** relation.

Within one thread, required evaluation order contributes **sequenced-before** edges. A release operation on an atomic object can **synchronize-with** an acquire operation that reads from the corresponding release sequence. Happens-before is the transitive closure of these and related edges. If one ordinary access happens-before another conflicting access, their order is defined; conflicting ordinary accesses without such ordering form a data race.

![A release store publishes preceding payload writes to a consumer whose acquire load observes the flag.](assets/happens-before-release-acquire.svg){fig-alt="Two-thread diagram showing sequenced-before, synchronizes-with, and transitive happens-before edges." width="96%"}

*Figure: original explanatory diagram based on the ISO C memory-model rationale [WG14 N1479](https://www.open-std.org/jtc1/sc22/wg14/www/docs/n1479.htm) and the POSIX memory-synchronization rules in [The Open Group Base Specifications](https://pubs.opengroup.org/onlinepubs/9799919799/basedefs/V1_chap04.html).*

<details>
<summary><strong>C11: safely publishing initialized data with release and acquire</strong></summary>

```c
#include <stdatomic.h>
#include <stdbool.h>
#include <stdio.h>

struct payload {
    int value;
    char message[32];
};

static struct payload shared_payload;
static atomic_bool ready = ATOMIC_VAR_INIT(false);

void producer(void) {
    // These ordinary writes are private until publication.
    shared_payload.value = 42;
    snprintf(shared_payload.message, sizeof shared_payload.message,
             "initialized");

    // Publish all preceding writes to a matching acquire reader.
    atomic_store_explicit(&ready, true, memory_order_release);
}

bool try_consume(void) {
    if (!atomic_load_explicit(&ready, memory_order_acquire)) {
        return false;
    }

    // Safe because the acquire observed the release publication.
    printf("%d %s\n", shared_payload.value, shared_payload.message);
    return true;
}
```

This protocol assumes one initialization publication and that no writer modifies the payload concurrently afterward. A reusable channel needs a state machine, ownership transfer, or another synchronization protocol.

</details>

The common C memory orders serve different contracts:

- `memory_order_relaxed` provides atomic modification order for that atomic object but no publication order for unrelated data;
- `memory_order_acquire` prevents later operations from moving before a successful acquiring observation;
- `memory_order_release` prevents earlier operations from moving after publication;
- `memory_order_acq_rel` combines both sides for a read-modify-write;
- `memory_order_seq_cst` adds a single total order among sequentially consistent atomic operations, making many protocols easier to reason about at a possible performance cost.

Use the weakest order only after writing the proof. For most application code, mutexes and standard concurrent containers encode a safer contract. Memory-order cleverness that nobody can review is a correctness liability.

### **Hardware Support for Synchronization**

Software synchronization ultimately needs a mechanism that cannot be interrupted by another processor observing a half-completed state transition. Architectures provide atomic read-modify-write instructions, cache-coherence protocols, and ordering instructions. Operating systems combine these with queues and scheduler support to avoid wasting CPUs during long waits.

#### **Disabling Interrupts and Its Limits**

On a uniprocessor kernel, disabling local interrupts can prevent interrupt handlers from interleaving with a short critical region on that CPU. It is not a general user-space lock and it does not stop another CPU. On a multiprocessor, CPU 1 continues accessing shared memory while CPU 0 has interrupts disabled.

Interrupt masking has strict limits:

- only privileged kernel code may control it directly;
- it protects against specific local execution contexts, not arbitrary remote CPUs;
- a long disabled interval increases interrupt and scheduling latency;
- nesting must restore the prior interrupt state, not blindly enable interrupts;
- non-maskable interrupts and machine-specific contexts may still exist;
- code must not sleep while relying on local interrupt disablement.

Linux distinguishes CPU-local protection, sleeping locks, and spinning locks. The exact semantics can change under `PREEMPT_RT`, which is why kernel code uses documented primitives such as `local_lock`, `spin_lock_irqsave`, and mutexes instead of inventing ad hoc interrupt protocols. The official [Linux lock-type rules](https://docs.kernel.org/locking/locktypes.html) define which categories may nest and which contexts may sleep.

#### **Test-and-Set, Compare-and-Swap, and Fetch-and-Add**

An atomic read-modify-write (RMW) reads a location, computes a result, and conditionally or unconditionally writes it as one indivisible modification relative to competing atomic operations.

- **Test-and-set** or exchange writes a locked value and returns the old value. It can build a minimal spinlock.
- **Compare-and-swap** (CAS) writes `desired` only if the location still equals `expected`. Failure tells an optimistic algorithm that another thread intervened.
- **Fetch-and-add** adds a value and returns the prior value. It naturally allocates unique sequence numbers and tickets.
- **Load-linked/store-conditional** is an alternative architecture interface: the store succeeds only if the reservation has not been invalidated.

![Atomic exchange, compare-and-swap, and fetch-and-add semantics, followed by a FIFO ticket-lock protocol.](assets/atomic-rmw-and-ticket-lock.svg){fig-alt="Three atomic read-modify-write operations and the acquire, wait, critical-section, and release stages of a ticket lock." width="98%"}

*Figure: original explanatory diagram based on the atomic-operation semantics in the [Linux atomic types documentation](https://docs.kernel.org/core-api/wrappers/atomic_t.html) and the lock construction discussion in [OSTEP Locks](https://pages.cs.wisc.edu/~remzi/OSTEP/threads-locks.pdf).*

A minimal C11 spinlock illustrates the role of acquire and release:

```c
#include <stdatomic.h>

typedef struct {
    atomic_flag held;
} spinlock_t;

#define SPINLOCK_INIT { ATOMIC_FLAG_INIT }

void spin_lock(spinlock_t *lock) {
    while (atomic_flag_test_and_set_explicit(&lock->held,
                                              memory_order_acquire)) {
        // In production, use an architecture pause hint and a bounded policy.
    }
}

void spin_unlock(spinlock_t *lock) {
    atomic_flag_clear_explicit(&lock->held, memory_order_release);
}
```

This is educational, not a complete production lock. It has no queue, timeout, owner validation, adaptive waiting, priority handling, or fairness guarantee. Under contention every waiter competes for one cache line. Real libraries and kernels use tuned implementations.

CAS also has an **ABA problem**: a location can change from value A to B and back to A, causing CAS to succeed even though the surrounding object changed. Tagged versions can detect some ABA cases, but safe memory reclamation remains essential when values are pointers.

#### **Memory Barriers**

Compilers and processors reorder independent operations to improve performance. A **compiler barrier** limits compiler motion but need not emit a CPU fence. A **CPU memory barrier** constrains how memory operations become observable to other agents. A full barrier orders more operation classes than an acquire or release barrier, but stronger is not automatically better: barriers cost performance and still require a correct pairing protocol.

The preferred reasoning unit is a synchronization operation:

```text
writer: ordinary initialization -> RELEASE publication
reader: ACQUIRE observation -> ordinary use
```

Both sides matter. A barrier on the writer alone does not force an unsynchronized reader to observe the intended order. Locks normally provide acquire semantics on successful lock and release semantics on unlock. Condition-variable waits synchronize through their mutex release and reacquisition. Use those documented guarantees instead of adding unexplained fences around racy data.

Kernel code sometimes needs explicit architecture-aware barriers for lock-free structures, device memory, interrupts, or DMA. The [Linux kernel memory-barrier guide](https://docs.kernel.org/core-api/wrappers/memory-barriers.html) distinguishes compiler, SMP, device, acquire, and release effects and warns that a barrier is a partial-ordering tool rather than a remote flush command.

### **Mutual-Exclusion Locks**

A mutual-exclusion lock gives one owner permission to execute a protected transition. The lock does not protect memory by proximity. Every access that relies on the invariant must follow the same protocol, and the protected object must not escape to code that ignores it.

Good lock documentation answers:

- which fields and invariants the lock protects;
- whether the lock may sleep and in which contexts it may be acquired;
- whether callbacks, allocation, I/O, or other locks are allowed while held;
- the global order relative to other locks;
- who owns and must release it;
- what happens if the owner exits or an operation times out.

POSIX mutexes have owner semantics: the successful locking thread is responsible for unlocking. Mutex types add behavior for recursion, error checking, robustness after owner death, priority inheritance, or priority ceilings. Those attributes are part of the design, not interchangeable performance flags.

#### **Spinlocks and Mutexes**

A **spinlock** keeps a waiter executing while it repeatedly checks lock state. A **mutex** can block a contending thread and let the scheduler run other work. The choice compares expected wait with the cost and legality of sleeping.

| Situation | Spin is plausible | Sleep is preferable |
|---|---:|---:|
| critical section is extremely short | yes | maybe |
| lock owner is currently running on another CPU | yes | maybe |
| wait may include I/O or an unbounded operation | no | yes |
| caller is an interrupt or non-sleeping kernel context | often required | illegal |
| CPU is oversubscribed or single-core | usually wasteful | yes |
| priority-sensitive workload | needs specialized design | PI mutex may help |

Linux user-space mutex implementations commonly use atomics for the uncontended path and futex operations when a thread must sleep. A futex wait is an atomic **compare-and-block** operation: the kernel sleeps the caller only if the futex word still has the expected value, closing the race between checking lock state and entering the wait queue.

![An adaptive mutex first tries an atomic user-space fast path, then may spin briefly or enter futex wait; unlock wakes a waiter only when needed.](assets/spin-mutex-futex-path.svg){fig-alt="Flowchart dividing user-space atomic mutex operations from kernel futex wait and wake paths." width="98%"}

*Figure: original explanatory diagram based on the Linux [`futex(2)`](https://man7.org/linux/man-pages/man2/futex.2.html) and [`futex(7)`](https://man7.org/linux/man-pages/man7/futex.7.html) interfaces. Exact lock-word states are library-specific.*

Waking a futex waiter does not immediately transfer the CPU or mutex. It makes a thread eligible; the scheduler decides when it runs, and it normally retries acquisition. This distinction prevents assumptions that `signal`, `post`, or `unlock` hands execution directly to one known waiter.

Kernel terminology requires care under `PREEMPT_RT`. On a conventional kernel, `spinlock_t` is a spinning, preemption-disabling lock. Under `PREEMPT_RT`, many `spinlock_t` instances map to priority-inheritance-aware sleeping locks, while `raw_spinlock_t` remains a true spinlock for low-level contexts. Code must follow the primitive's documented context rules instead of reasoning from its name alone.

#### **Lock Granularity and Contention**

A coarse lock protects a large object or subsystem with a simple invariant. It is easier to review and usually has fewer lock-order edges, but unrelated operations serialize. Fine-grained locks increase potential parallelism while increasing metadata, cache traffic, ordering complexity, and the chance of accessing fields under the wrong lock.

**Contention** is not merely the number of threads. It depends on arrival rate, hold time, write frequency, cache-line placement, and scheduling. A useful first approximation is that reducing time inside a highly contended critical section often matters more than making lock acquisition a few instructions faster.

Practical transformations include:

- move slow computation outside the critical section while preserving a valid snapshot;
- shard a table and lock each bucket or partition independently;
- use per-CPU counters and aggregate when exact instantaneous totals are unnecessary;
- separate read-mostly metadata from frequently written state;
- batch updates to reduce handoffs;
- avoid calling unknown callbacks while holding an internal lock;
- avoid blocking I/O, page faults, and allocation with unsafe flags while holding a spinlock.

Fine-grained locking requires a written order. If an operation needs two account locks, sort by a stable account ID and always acquire in that order. Comparing unrelated C pointers with `<` is not a portable ordering rule; use an explicit key.

False sharing can make independent locks contend in hardware when they occupy one cache line. Conversely, padding every object wastes cache capacity. Measure lock wait time, hold time, handoffs, and cache misses before redesigning. A profiler showing high CPU usage in a spin loop demands a different repair from a trace showing threads asleep behind a long mutex owner.

### **Semaphores, Condition Variables, and Monitors**

A **counting semaphore** stores a nonnegative number of available permits. `wait` decrements when a permit exists or blocks otherwise; `post` increments and may wake a waiter. A binary semaphore can have values zero or one, but it is not automatically a mutex: semaphores generally lack ownership, so a different thread may post, and priority-inheritance behavior may differ.

Semaphores are useful for resource counts, admission limits, and event handoff. They can encode producer-consumer capacity with `empty` and `full` counts, but a separate mutex is still needed for multi-field buffer state. Errors such as one extra `post` silently alter the permit invariant.

A **condition variable** has different semantics. It lets a thread wait until shared state protected by a mutex may satisfy a predicate. It does not remember arbitrary application state, and `signal` is not the predicate. The canonical rule is:

```c
pthread_mutex_lock(&mutex);
while (!predicate()) {
    pthread_cond_wait(&condition, &mutex);
}
use_state_that_satisfies_predicate();
pthread_mutex_unlock(&mutex);
```

`pthread_cond_wait` atomically releases the mutex and begins waiting, then reacquires the mutex before returning. The atomic transition avoids a **lost wakeup** between checking the predicate and sleeping. The loop is still required because wakeups may be spurious, several waiters may race for one state change, or another thread may consume the condition before this waiter reacquires the mutex.

![Consumer wait and producer signal protocol around one mutex-protected predicate.](assets/condition-variable-protocol-animated.svg){fig-alt="Animated two-thread flowchart showing predicate check, atomic mutex release and wait, state change, signal, reacquisition, and recheck." width="98%"}

*Figure: original explanatory diagram based on the POSIX condition-variable memory synchronization rules in [The Open Group Base Specifications](https://pubs.opengroup.org/onlinepubs/9799919799/basedefs/V1_chap04.html) and [OSTEP Condition Variables](https://pages.cs.wisc.edu/~remzi/OSTEP/threads-cv.pdf).*

Modify the predicate while holding the associated mutex. Signaling while the lock is held often makes the protocol easier to review; unlocking afterward lets a woken waiter eventually acquire and inspect the new state. `signal` wakes at least an appropriate waiter according to the interface, whereas `broadcast` makes all waiters eligible and is needed when a transition can satisfy different predicates or when closing a shared object. Broadcasting on every update can create a thundering herd.

A **monitor** packages shared state, operations, mutual exclusion, and condition variables behind one abstraction. The key benefit is ownership of the invariant: callers cannot update fields without passing through monitor methods. Most POSIX condition-variable reasoning follows Mesa-style semantics, where a signal makes a waiter runnable but does not transfer the lock immediately; therefore the waiter must recheck.

### **Barriers and Read-Write Coordination**

A **barrier** divides execution into phases. Each participating thread reports arrival and waits until the configured number has arrived; then the generation opens and all can enter the next phase. A reusable barrier needs both an arrival count and a generation number so a fast thread returning for phase 2 cannot be mistaken for a late arrival in phase 1.

A thread that exits, throws, or blocks forever before arrival can strand every peer. The participant count and cancellation protocol must therefore match the task lifecycle. Barriers suit bulk-synchronous algorithms; they are a poor fit when tasks progress independently or have highly skewed work.

A **read-write lock** admits multiple readers or one exclusive writer. It helps when read critical sections are substantial, reads dominate, and the protected data truly supports concurrent reads. It can be slower than a mutex for short sections because reader acquisition updates shared accounting and can generate cache traffic.

![A generation barrier holds early arrivals before phase two, while a readers-writer lock alternates shared and exclusive admission modes.](assets/barrier-and-rwlock.svg){fig-alt="Two-panel diagram of a reusable phase barrier and readers-writer lock admission policies." width="98%"}

*Figure: original explanatory diagram created for this chapter from POSIX barrier and read-write-lock semantics.*

Reader-writer fairness is a policy choice. Reader preference maximizes read admission but can starve a writer under continuous arrivals. Writer preference bounds writer delay but can create reader latency spikes. Phase-fair approaches alternate cohorts with additional bookkeeping. The API name alone does not reveal the policy, so portability-sensitive code should not assume one.

Sequence counters provide another read-mostly pattern: a writer increments a sequence around an update, while a reader retries if it observed an odd or changed sequence. Readers do not block writers, but the read operation must be repeatable and protected objects cannot be freed during a retry. A sequence counter is not a general replacement for a read-write lock.

### **Classical Synchronization Problems**

Classical problems are compact protocol tests. Their value is not memorizing one semaphore arrangement; it is learning to identify predicates, ownership, wait edges, fairness, and shutdown behavior that reappear in queues, caches, filesystems, databases, and device drivers.

#### **Producer-Consumer**

In a bounded producer-consumer system, producers require `count < capacity`, consumers require `count > 0`, and both update the same buffer representation. Mutual exclusion protects `head`, `tail`, `count`, and slots. Two condition variables distinguish why a thread waits: `not_full` for producers and `not_empty` for consumers.

![Bounded ring buffer with producer and consumer predicates, mutex-protected representation, and condition-variable wakeups.](assets/bounded-buffer-producer-consumer.svg){fig-alt="Producer and consumer protocol around a bounded ring buffer with not-full and not-empty predicates." width="98%"}

*Figure: original explanatory diagram created for this chapter. The pipe analogy follows the blocking and close semantics of POSIX pipes.*

<details>
<summary><strong>C: a bounded pthread queue with close semantics</strong></summary>

```c
#include <errno.h>
#include <pthread.h>
#include <stdbool.h>
#include <stddef.h>

enum { QUEUE_CAPACITY = 8 };

struct queue {
    int items[QUEUE_CAPACITY];
    size_t head;
    size_t tail;
    size_t count;
    bool closed;
    pthread_mutex_t mutex;
    pthread_cond_t not_empty;
    pthread_cond_t not_full;
};

#define QUEUE_INITIALIZER {                         \
    .head = 0, .tail = 0, .count = 0, .closed = false, \
    .mutex = PTHREAD_MUTEX_INITIALIZER,             \
    .not_empty = PTHREAD_COND_INITIALIZER,           \
    .not_full = PTHREAD_COND_INITIALIZER             \
}

// Returns 0, or EPIPE if the queue has been closed.
int queue_put(struct queue *queue, int value) {
    pthread_mutex_lock(&queue->mutex);

    while (queue->count == QUEUE_CAPACITY && !queue->closed) {
        pthread_cond_wait(&queue->not_full, &queue->mutex);
    }

    if (queue->closed) {
        pthread_mutex_unlock(&queue->mutex);
        return EPIPE;
    }

    queue->items[queue->tail] = value;
    queue->tail = (queue->tail + 1) % QUEUE_CAPACITY;
    queue->count++;

    pthread_cond_signal(&queue->not_empty);
    pthread_mutex_unlock(&queue->mutex);
    return 0;
}

// Returns false only when the closed queue has become empty.
bool queue_get(struct queue *queue, int *value) {
    pthread_mutex_lock(&queue->mutex);

    while (queue->count == 0 && !queue->closed) {
        pthread_cond_wait(&queue->not_empty, &queue->mutex);
    }

    if (queue->count == 0) {
        pthread_mutex_unlock(&queue->mutex);
        return false;
    }

    *value = queue->items[queue->head];
    queue->head = (queue->head + 1) % QUEUE_CAPACITY;
    queue->count--;

    pthread_cond_signal(&queue->not_full);
    pthread_mutex_unlock(&queue->mutex);
    return true;
}

void queue_close(struct queue *queue) {
    pthread_mutex_lock(&queue->mutex);
    queue->closed = true;

    // Every waiter must re-evaluate either data availability or closure.
    pthread_cond_broadcast(&queue->not_empty);
    pthread_cond_broadcast(&queue->not_full);
    pthread_mutex_unlock(&queue->mutex);
}
```

Production code should check every pthread return value, define cancellation behavior, destroy primitives only after all users have stopped, and document whether already-buffered items remain readable after close. Those lifecycle rules are part of concurrent correctness.

</details>

The shell pipeline follows the same shape but lets the kernel own it. `cat` is a producer, `grep` is a consumer, pipe capacity supplies backpressure, and closing file descriptors communicates end-of-stream. Process isolation reduces accidental shared-memory races, but the pipe implementation inside the kernel still needs correct synchronization.

#### **Readers-Writers**

The readers-writers problem asks how to admit operations when reads can coexist but writes require exclusivity. The invariant is not simply "many readers are safe." Readers must be side-effect free with respect to protected state, object lifetime must remain valid, and no hidden cache or statistics write may violate the assumption.

A POSIX interface makes the access mode visible:

```c
pthread_rwlock_rdlock(&table_lock);
const struct record *record = lookup_without_mutation(key);
copy_record_for_caller(record);  // Do not return an unprotected pointer.
pthread_rwlock_unlock(&table_lock);

pthread_rwlock_wrlock(&table_lock);
insert_or_replace_record(key, value);
pthread_rwlock_unlock(&table_lock);
```

Three policies are commonly compared:

- reader preference admits arriving readers while readers are active, maximizing read concurrency but risking writer starvation;
- writer preference blocks new readers once a writer waits, bounding writer delay but extending reader queues;
- fair or phased admission orders cohorts, trading throughput for predictability.

The returned-pointer comment is crucial. Unlocking before a caller finishes using a record allows a writer to delete or replace it. Copying, reference counting, immutable versions, or RCU can solve the lifetime problem; a read lock held only during lookup cannot.

#### **Dining Philosophers**

The dining philosophers problem represents threads that each need two neighboring resources before entering a critical action. If every philosopher acquires the left fork and then waits for the right, all can hold one resource while waiting in a cycle.

![Five dining philosophers share one fork with each neighbor.](assets/dining-philosophers.svg){fig-alt="Circular arrangement of five philosophers and five shared forks." width="44%"}

*Figure source: DnetSvg and Allen3, [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Dining_philosophers.svg), released to the public domain. The original SVG is stored locally.*

Several repairs break different assumptions:

- impose a total order on forks and always acquire the lower-numbered fork first, breaking circular wait;
- use a waiter or semaphore with at most four permits, preventing all five from holding one fork simultaneously;
- make one philosopher acquire in the opposite order, also breaking the symmetric cycle;
- use try-lock, release partial holdings on failure, and retry with randomized or queued backoff;
- allocate both resources atomically through a monitor.

Deadlock freedom is not the whole result. A try-lock loop can livelock if every philosopher moves in synchrony. An unfair waiter can starve one participant. A global waiter is easy to prove but reduces concurrency because disjoint pairs might otherwise eat simultaneously. The preferred solution depends on the required progress guarantee and contention pattern.

### **Deadlock**

A **deadlock** is a state in which a set of participants cannot progress because each waits for an event that only another blocked participant in the set can cause. The threads may consume no CPU, so low utilization can be a symptom. Deadlock differs from a slow lock holder: in the latter case an enabled action still exists that can release the resource.

#### **Necessary Conditions and Resource-Allocation Graphs**

The Coffman conditions are jointly necessary for classic reusable-resource deadlock:

1. **mutual exclusion:** at least one resource cannot be shared concurrently;
2. **hold and wait:** a participant holds one resource while requesting another;
3. **no preemption:** resources are released voluntarily rather than forcibly taken;
4. **circular wait:** a directed cycle exists in which each participant waits for the next.

A **resource-allocation graph** has process nodes and resource nodes. A process-to-resource edge is a request; a resource-instance-to-process edge is an assignment.

![A resource-allocation graph containing process requests and resource assignments.](assets/resource-allocation-graph.svg){fig-alt="Graph showing processes, resource instances, request edges, and assignment edges." width="58%"}

*Figure source: Martin Thoma, [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Resource-allocation-graph.svg), licensed under [CC BY 3.0](https://creativecommons.org/licenses/by/3.0/). The original SVG is stored locally without modification.*

With one instance of each resource type, a cycle is necessary and sufficient for deadlock. With multiple instances, a cycle is necessary but may not be sufficient because another free instance or completing process may break the wait. A **wait-for graph** removes resource nodes and draws `A -> B` when A waits for a resource held by B; cycles then reveal candidate deadlocks under the modeled assumptions.

The smallest lock-order inversion is:

```text
Thread A: lock(X) -> lock(Y)
Thread B: lock(Y) -> lock(X)
```

If A owns X and B owns Y before their second acquisitions, neither can continue. The failure may occur once in a million runs, but the dependency cycle exists from the two code paths alone. Linux `lockdep` records observed lock-class orderings and detects such cycles without requiring the complete deadlocking schedule to occur.

#### **Prevention, Avoidance, Detection, and Recovery**

**Prevention** designs away at least one necessary condition:

- make resources shareable where semantics allow immutable or versioned access;
- request all required resources at once, reducing hold-and-wait but lowering utilization;
- release acquired resources when a later try-lock fails, introducing retry and possible livelock;
- impose a global resource order, breaking circular wait;
- use preemptible or revocable resources when rollback is safe.

Lock ordering is the most common software technique. Assign every lock class or object a stable rank and permit acquisition only in increasing order. Dynamic sets should be sorted before locking. The rule must include callbacks and error paths; acquiring an unranked lock from inside a ranked critical section reintroduces an unknown edge.

**Avoidance** examines each request and grants it only if the resulting state remains **safe**, meaning some order still exists in which every participant can obtain its declared maximum and finish. The Banker's algorithm models available resources, current allocations, and maximum claims. It is valuable conceptually but uncommon for ordinary mutexes because programs rarely declare trustworthy maximum future lock needs and requests arrive at instruction-level frequency.

An unsafe state is not yet deadlocked. It has lost the guarantee that every future legal demand can be satisfied. A deadlocked state has no completion sequence for the involved waiters.

**Detection** allows waits and periodically searches a wait-for graph or resource state. It works when the system can observe ownership and waiting accurately. Distributed systems make detection harder because graph information is delayed and no observer has an instantaneous global state.

**Recovery** may terminate a victim, roll back a transaction, revoke a resource, restart a component, or rely on a timeout. Recovery must preserve external consistency. Killing a thread while it holds an ordinary in-process mutex can leave data corrupted and waiters blocked. Robust mutexes can report owner death, but the next owner must repair protected state and explicitly mark it consistent.

Database schemes such as wait-die and wound-wait use transaction ages to prevent cycles and abort work safely because transactions provide rollback. That assumption does not transfer automatically to arbitrary kernel or application critical sections.

### **Starvation, Livelock, and Priority Inversion**

These progress failures look similar from a user's perspective but require different repairs:

| Failure | What threads are doing | Typical cause | Typical repair |
|---|---|---|---|
| deadlock | blocked in a dependency cycle | inconsistent resource order | break a Coffman condition or recover |
| starvation | system progresses but one participant does not | unfair admission or strict priority | FIFO queue, aging, bounded handoff |
| livelock | participants execute but repeatedly undo or retry | symmetric conflict response | backoff, asymmetry, arbitration |
| priority inversion | urgent thread waits indirectly behind lower priority | low-priority resource owner is preempted | priority inheritance or ceiling |

Priority inversion involves at least three roles. Low-priority L owns a mutex; high-priority H blocks on it; medium-priority M does not need the mutex but repeatedly preempts L. H is then delayed by M even though H outranks M.

![Without inheritance, medium-priority work delays the low-priority lock owner and therefore the high-priority waiter; inheritance temporarily boosts the owner.](assets/priority-inversion-inheritance.svg){fig-alt="Two scheduling timelines compare priority inversion before and after priority inheritance." width="98%"}

*Figure: original explanatory diagram based on Linux [priority-inheritance futex](https://man7.org/linux/man-pages/man2/futex.2.html) and [RT-mutex](https://docs.kernel.org/locking/rt-mutex-design.html) semantics.*

**Priority inheritance** temporarily propagates the highest waiting priority to the owner, potentially through a chain, so it can run and release the resource. **Priority ceiling** assigns each resource a ceiling related to its possible users and can bound blocking under a stricter protocol. Neither removes deadlock or long critical sections. Inheritance also requires ownership, which is one reason a generic counting semaphore cannot provide the same semantics cleanly.

Starvation and livelock should be measured with per-request wait distributions and retry counts. Aggregate throughput can remain high while one tenant receives no service. Fair queueing, ticket locks, aging, randomized backoff, and admission limits are policy tools; each trades some throughput or simplicity for a stronger progress claim.

### **Lock-Free Techniques and Read-Copy-Update**

Nonblocking algorithms replace ownership waits with atomic state transitions and retries. The terms describe progress guarantees, not whether source code contains the word `lock`:

- **obstruction-free:** an operation finishes if it eventually runs without interference;
- **lock-free:** in a finite number of system-wide steps, some operation finishes;
- **wait-free:** every operation finishes within a bounded number of its own steps.

A CAS loop that increments a counter is typically lock-free but not wait-free because one thread can repeatedly lose:

```c
#include <stdatomic.h>

void increment_with_cas(atomic_long *counter) {
    long observed = atomic_load_explicit(counter, memory_order_relaxed);

    while (!atomic_compare_exchange_weak_explicit(
        counter,
        &observed,
        observed + 1,
        memory_order_relaxed,
        memory_order_relaxed)) {
        // On failure, observed is replaced with the value that defeated us.
    }
}
```

For an independent counter, `atomic_fetch_add_explicit` is simpler. The C implementation may use an internal lock for an atomic type that is not lock-free; `atomic_is_lock_free` reports the implementation property. Algorithmic lock freedom also depends on object lifetime and every helper it calls.

The hardest part of a lock-free pointer structure is often **memory reclamation**. Removing a node from a list does not prove that another thread has stopped reading it. Freeing immediately creates use-after-free; never freeing leaks memory. Hazard pointers, epoch-based reclamation, reference counting, and RCU establish when old objects can be reclaimed. ABA tags alone do not solve dangling readers.

**Read-Copy-Update** (RCU) targets read-mostly structures. An updater creates or modifies a replacement, atomically publishes a new pointer, waits for a **grace period** during which all pre-existing read-side critical sections finish, and only then reclaims the old version.

![Readers that began before publication may use the old object; after a grace period the updater can reclaim it safely.](assets/rcu-grace-period.svg){fig-alt="RCU timeline with old and new readers, publication, grace period, and deferred reclamation." width="98%"}

*Figure: original explanatory diagram based on the official Linux [RCU concepts](https://docs.kernel.org/RCU/whatisRCU.html) and [RCU Handbook](https://docs.kernel.org/RCU/index.html).*

A conceptual kernel pattern is:

```c
/* Reader */
rcu_read_lock();
item = rcu_dereference(current_item);
use_immutable_fields(item);
rcu_read_unlock();

/* Writer, with separate serialization among writers */
new_item = copy_and_modify(old_item);
rcu_assign_pointer(current_item, new_item);
synchronize_rcu();
free(old_item);
```

Readers may see the old or new version, but each must see a valid version throughout its protected lifetime. RCU does not make in-place mutation safe, serialize competing writers, or permit pointers to escape beyond the read-side lifetime. Linux has several RCU flavors with different quiescent-state and blocking rules; code must use the flavor appropriate to its context.

RCU is powerful when reads dominate, stale snapshots are acceptable, and updates can afford copying and deferred reclamation. Mutexes remain clearer for balanced read-write workloads, multi-field transactions, and code whose readers need to block or retain references independently.

### **Diagnosing and Testing Concurrent Code**

Concurrency testing explores executions; it does not prove that all executions are safe. Effective practice combines an explicit synchronization contract, compiler and runtime instrumentation, schedule variation, traces, and regression tests.

![Workflow from invariants and executable checks through race detectors, schedule exploration, trace minimization, repair, and regression.](assets/concurrency-diagnostics-workflow.svg){fig-alt="Cyclic workflow for diagnosing concurrent correctness and progress failures." width="98%"}

*Figure: original explanatory diagram based on LLVM [ThreadSanitizer](https://clang.llvm.org/docs/ThreadSanitizer.html) and Linux [lockdep](https://docs.kernel.org/locking/lockdep-design.html) workflows.*

ThreadSanitizer instruments memory accesses and synchronization to report data races:

```bash
clang -O1 -g -fsanitize=thread -fno-omit-frame-pointer \
  race.c -pthread -o race-tsan
./race-tsan
```

Compile as much of the program as possible with the sanitizer. Uninstrumented libraries, custom assembly, and unsupported synchronization can cause missed reports or false positives. A clean run means no race occurred in explored executions under the detector; it is not a proof that no race exists.

Useful testing dimensions include:

- one CPU versus many CPUs, because preemption races and true parallel races differ;
- low and high contention, varying thread counts and queue capacity;
- randomized yields or delays around synchronization boundaries;
- repeated startup, shutdown, cancellation, timeout, and error paths;
- allocation failure and owner termination where recovery is supported;
- long runs under sanitizers plus shorter performance runs without instrumentation;
- architectures with weaker ordering than the developer's primary machine.

For Linux kernel code, `lockdep` builds lock-class dependency graphs and reports illegal ordering and interrupt-context combinations; KCSAN samples accesses to find races; lock torture and RCU torture stress specific primitives. Each tool targets a property. A race detector does not verify a business invariant, and a lock-order checker does not establish bounded waiting.

When a program hangs, collect all thread stacks and identify waits and owners. Draw a wait-for graph rather than guessing from the last log line. Record monotonic event IDs, object identities, lock names, predicate values, and generation numbers. Logging inside a critical section changes timing, so preserve enough evidence to reconstruct order without assuming logs are neutral.

Assertions should encode the invariant while the required lock is held:

```c
pthread_mutex_lock(&queue->mutex);
assert(queue->count <= QUEUE_CAPACITY);
assert(queue_representation_is_consistent(queue));
pthread_mutex_unlock(&queue->mutex);
```

For small protocols, model checking or exhaustive state exploration can examine schedules that stress testing rarely reaches. The model must still match implementation semantics, especially atomicity and memory ordering. The best debugging artifact is a minimized execution that names the two conflicting events and the missing synchronization edge.

### **Comparison and Summary**

| Mechanism | Ownership | Waiting behavior | Best fit | Frequent mistake |
|---|---|---|---|---|
| mutex | one owner | normally blocks under contention | multi-field invariants and ordinary critical sections | inconsistent coverage or blocking while held |
| spinlock | one owner | consumes CPU while waiting | extremely short non-sleeping kernel paths | spinning on an oversubscribed CPU |
| semaphore | permit count, usually no owner | blocks when no permit | capacity and resource admission | treating permits as protected-state ownership |
| condition variable | paired with mutex and predicate | atomic unlock-sleep-relock | waiting for state transitions | using `if`, or treating signal as stored state |
| barrier | fixed participant generation | waits for phase cohort | bulk-synchronous phases | participant exits before arrival |
| read-write lock | shared readers or one writer | policy-dependent | substantial read-mostly critical sections | assuming readers are free or writer-fair |
| C atomic RMW | no blocking owner | retry or direct operation | independent counters and carefully proven lock-free state | confusing atomicity with publication ordering |
| sequence counter | one serialized writer, retrying readers | readers retry | small read-mostly snapshots | pointers or expensive non-repeatable reads |
| RCU | read-side lifetime plus updater protocol | cheap reads, deferred reclamation | read-mostly versioned structures | freeing before grace period or mutating in place |

A robust design process is:

1. state the shared object's invariant and lifetime;
2. mark every operation that can temporarily violate it;
3. define the atomic transition or linearization point;
4. define visibility with lock handoff or explicit happens-before edges;
5. write waiting as predicates over protected state;
6. document lock order, ownership, context, and error paths;
7. state the progress guarantee and scheduler assumptions;
8. test with instrumentation and adversarial schedules;
9. measure contention before replacing a clear lock with a complex nonblocking design.

Several misconceptions can now be rejected. `volatile` is not thread synchronization. Atomic access to a flag does not automatically publish unrelated data. A condition signal does not mean its predicate remains true. Deadlock freedom does not imply starvation freedom. Lock-free does not mean every thread finishes promptly, and RCU does not mean old memory can be freed immediately.

The central idea is a protocol, not a primitive. Correct synchronization defines which states are legal, which transitions are indivisible, how one thread's work becomes visible to another, and why every required participant can eventually progress. Locks, atomics, condition variables, semaphores, barriers, and RCU are tools for expressing that proof.
